# EC売上・顧客分析

架空のECサイトデータを使い、売上トレンド・カテゴリ別収益・顧客LTV・新規/既存比較・売上集中度を分析します。

---

**実行環境:** MySQL 8.0 / Python 3 / pandas  
**DB:** `sql_portfolio`（`SETUP.md` の手順で事前に構築）

## セットアップ

In [1]:
import mysql.connector
import pandas as pd
from IPython.display import display, HTML

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

con = mysql.connector.connect(
    host='localhost', user='root', password='',
    database='sql_portfolio',
    charset='utf8mb4'
)

def run(sql):
    return pd.read_sql(sql, con)


---

## 分析クエリ

### 01. 月次売上推移・前月比分析

**ビジネス課題:** 売上の月次トレンドと前月比を定量化し、異常値の早期検知や施策効果の評価に活用する。

**使用テクニック:** CTE, `LAG()` ウィンドウ関数, `DATE_FORMAT`, `NULLIF`

In [2]:
sql = '''
WITH monthly_sales AS (
    SELECT
        DATE_FORMAT(o.order_date, '%Y-%m') AS ym,
        SUM(oi.quantity * oi.unit_price)   AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status IN ('paid','shipped')
    GROUP BY DATE_FORMAT(o.order_date, '%Y-%m')
)
SELECT
    ym,
    revenue,
    LAG(revenue) OVER (ORDER BY ym) AS prev_revenue,
    ROUND(
        (revenue - LAG(revenue) OVER (ORDER BY ym)) /
        NULLIF(LAG(revenue) OVER (ORDER BY ym), 0) * 100,
        1
    ) AS mom_change_pct
FROM monthly_sales
ORDER BY ym
'''
df = run(sql)
display(df)


C:\Users\willi\AppData\Local\Temp\ipykernel_30248\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,ym,revenue,prev_revenue,mom_change_pct
0,2024-04,"7,500.00",NaN,NaN
1,2024-05,"4,500.00","7,500.00",-40.00
2,2024-06,"1,200.00","4,500.00",-73.30


### 02. カテゴリ別売上ランキング

**ビジネス課題:** 売上貢献度の高い商品カテゴリを特定し、仕入れやプロモーション戦略の優先順位付けに活用する。

**使用テクニック:** 4テーブル JOIN, `GROUP BY`, `ORDER BY DESC`

In [3]:
sql = '''
SELECT
    c.category_name,
    SUM(oi.quantity * oi.unit_price) AS revenue
FROM order_items oi
JOIN orders     o  ON oi.order_id   = o.order_id
JOIN products   p  ON oi.product_id = p.product_id
JOIN categories c  ON p.category_id = c.category_id
WHERE o.status IN ('paid','shipped')
GROUP BY c.category_id, c.category_name
ORDER BY revenue DESC
LIMIT 10
'''
df = run(sql)
display(df)


C:\Users\willi\AppData\Local\Temp\ipykernel_30248\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,category_name,revenue
0,Electronics,"7,300.00"
1,Clothing,"4,500.00"
2,Food,"1,400.00"


### 03. 顧客LTV（生涯価値）分析

**ビジネス課題:** 顧客ごとの累計売上・注文回数・平均注文額を算出し、VIP顧客の識別やCRM施策の優先度判定に活用する。

**使用テクニック:** CTE（注文単位の小計を先に集計）, JOIN, 集計関数

In [4]:
sql = '''
WITH order_amounts AS (
    SELECT
        o.order_id,
        o.customer_id,
        SUM(oi.quantity * oi.unit_price) AS order_amount
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status IN ('paid','shipped')
    GROUP BY o.order_id, o.customer_id
)
SELECT
    c.customer_id,
    c.customer_name,
    COUNT(oa.order_id)             AS order_count,
    SUM(oa.order_amount)           AS lifetime_value,
    ROUND(AVG(oa.order_amount), 2) AS avg_order_value
FROM customers c
JOIN order_amounts oa ON c.customer_id = oa.customer_id
GROUP BY c.customer_id, c.customer_name
ORDER BY lifetime_value DESC
LIMIT 50
'''
df = run(sql)
display(df)


C:\Users\willi\AppData\Local\Temp\ipykernel_30248\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,customer_id,customer_name,order_count,lifetime_value,avg_order_value
0,1,田中太郎,2,"5,000.00","2,500.00"
1,3,佐藤一郎,1,"4,500.00","4,500.00"
2,2,鈴木花子,2,"3,700.00","1,850.00"


### 04. 新規顧客 vs 既存顧客の月次売上比較

**ビジネス課題:** 月ごとの売上が新規獲得とリピートのどちらに依存しているかを把握し、マーケティング予算の配分判断に用いる。

**使用テクニック:** CTE（初回注文日を特定）, `CASE` 式による分類

In [5]:
sql = '''
WITH first_orders AS (
    SELECT
        customer_id,
        MIN(order_date) AS first_order_date
    FROM orders
    WHERE status IN ('paid','shipped')
    GROUP BY customer_id
),
order_with_flag AS (
    SELECT
        o.*,
        CASE
            WHEN DATE(o.order_date) = DATE(f.first_order_date) THEN 'new'
            ELSE 'existing'
        END AS customer_type
    FROM orders o
    JOIN first_orders f ON o.customer_id = f.customer_id
    WHERE o.status IN ('paid','shipped')
)
SELECT
    DATE_FORMAT(o.order_date, '%Y-%m') AS ym,
    customer_type,
    SUM(oi.quantity * oi.unit_price)   AS revenue
FROM order_with_flag o
JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY DATE_FORMAT(o.order_date, '%Y-%m'), customer_type
ORDER BY ym, customer_type
'''
df = run(sql)
display(df)


C:\Users\willi\AppData\Local\Temp\ipykernel_30248\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,ym,customer_type,revenue
0,2024-04,existing,"3,200.00"
1,2024-04,new,"4,300.00"
2,2024-05,new,"4,500.00"
3,2024-06,existing,"1,200.00"


### 05. 売上トップ10%顧客の集中度分析（パレート分析）

**ビジネス課題:** 売上上位顧客への依存度を定量化し、顧客基盤のリスク評価や優良顧客向け施策の費用対効果を検討する。

**使用テクニック:** CTE（3段階）, `NTILE()` ウィンドウ関数によるデシル分割, 条件付き集計

In [6]:
sql = '''
WITH customer_ltv AS (
    SELECT
        o.customer_id,
        SUM(oi.quantity * oi.unit_price) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status IN ('paid','shipped')
    GROUP BY o.customer_id
),
ranked AS (
    SELECT
        customer_id,
        revenue,
        NTILE(10) OVER (ORDER BY revenue DESC) AS decile
    FROM customer_ltv
),
agg AS (
    SELECT
        SUM(revenue) AS total_revenue,
        SUM(CASE WHEN decile = 1 THEN revenue ELSE 0 END) AS top10_revenue
    FROM ranked
)
SELECT
    total_revenue,
    top10_revenue,
    ROUND(top10_revenue / total_revenue * 100, 1) AS top10_share_pct
FROM agg
'''
df = run(sql)
display(df)


C:\Users\willi\AppData\Local\Temp\ipykernel_30248\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,total_revenue,top10_revenue,top10_share_pct
0,"13,200.00","5,000.00",37.90


---

In [7]:
con.close()
print('接続を閉じました。')

接続を閉じました。
